# Capstone C: Ship a Fine-Tuned-Transformer Chatbot

This is the finale. Across this course you went from tools-first NLP, to embeddings as
geometry, to an MLP on word2vec features, and finally to fine-tuning real transformers:
DistilBERT (C9) and T5 (C10). Each of those ended in a small Gradio chatbot.

Now you own one. You will:

1. Pick ONE architecture - an encoder-only classifier (C9 style) OR an encoder-decoder
   Q&A model (C10 style).
2. Choose a small, public dataset that fits the use case.
3. Fine-tune a pretrained model on it.
4. Evaluate it honestly on a held-out split.
5. Ship it inside a Gradio chatbot.
6. Write up what works, where it breaks, and what production would need.

No new theory. This is the engineering decision and the delivery. You have built every
piece before - here you assemble it unaided.

A note on how this notebook is gated. Because this is an "assemble unaided" capstone, each
lab ends in a hard `assert` so an unfinished lab stops right there instead of silently
breaking the finale - the asserts are intentional gates, not hints. After each path's labs
there is also a PROVIDED safety-net cell: if you had to skip a lab, it backfills a working
(but untrained) fallback model and inference function so the shared save/reload and the
Gradio ship cells still run end to end. Aim to finish the labs; the safety-nets only exist
so the chatbot finale is always reachable.

## The scenario

Same support platform you have worked all course. Two prototypes exist from the last two
sessions:

- The C9 DistilBERT ticket-tagger: reads a customer message, returns POSITIVE / NEGATIVE
  with a confidence score.
- The C10 T5 answer-writer: reads a question plus a help-doc snippet and WRITES an answer.

Leadership: "Great demos. Now you own one. Pick the architecture our product needs next,
ship a clean version on a dataset you can justify, evaluate it, and tell us honestly where
it breaks before a customer finds out."

That is the capstone. One path. Real fine-tune. Honest evaluation. A shipped chatbot. A
write-up. The whole course pointed here.

## How to choose your path

The single most important capstone decision is architecture. The rule you learned in C9
and C10:

- Encoder-only (Path A, DistilBERT): the output is a LABEL from a fixed set. Use it when
  the job is routing, tagging, sentiment, intent, or any "which bucket?" question. Fast,
  cheap, easy to evaluate (accuracy / F1), and it CANNOT make up words - it can only pick
  a class. It also cannot explain or answer in prose.
- Encoder-decoder (Path B, T5): the output is free TEXT the model writes. Use it when the
  job is to answer, summarize, or rewrite. More powerful, but it can HALLUCINATE - a
  SQuAD-trained T5 mostly copies spans from the context and will confidently invent an
  answer when the context does not contain one. Harder to evaluate, and you must ground it.

Decision rule: if a human could solve the task by picking from a short menu, choose Path A.
If the task genuinely requires writing new text, choose Path B, and plan for grounding.

Pick now. You will only run one path's cells.

**Diagram: encoder-only vs encoder-decoder - which path?**

![Encoder-only vs encoder-decoder decision](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/3-Transfer-Chatbot/diagrams/capstone-c/encoder-decoder-shapes.png)

In [ ]:
# Install the course stack, pinned. Run this FIRST in Colab.
# numpy<2 is required course-wide; this DOWNGRADES numpy (Colab ships numpy 2.x), so you
# MUST restart the runtime after this cell (Runtime -> Restart session), then run every
# other cell. transformers is pinned to 4.57.1; do NOT let it resolve to 5.x.
!pip install -q "transformers==4.57.1" "datasets>=2.19,<3" "numpy<2" \
    evaluate accelerate gradio

# After restart, re-run from the next cell down (do NOT re-run this install cell).
print("Install done. Now: Runtime -> Restart session, then continue from the next cell.")

# Download the small English spaCy pipeline (model weights, separate from pip).
!python -m spacy download en_core_web_sm

# TextBlob/NLTK tokenizer data (punkt_tab) for sentence/word tokenization.
import nltk
nltk.download('punkt_tab')


In [ ]:
# Imports grouped by purpose. These cover BOTH paths; you only call the ones your path uses.
import random
import numpy as np
import torch

from datasets import load_dataset

# Encoder-only (Path A) pieces:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
# Encoder-decoder (Path B) pieces:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

import evaluate

# Reproducibility (same SEED block as C9).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device: a torch.device object. For HF pipeline(device=...) convert to int 0 (cuda) or -1 (cpu).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Transformers stack ready. Device:", device)

## Checkpoint: choose your path

You are about to commit to one of two paths. Read both, then set PATH in the next cell.

PATH A - Encoder-only classification chatbot (C9 style)
  Model:   distilbert-base-uncased
  Output:  a label + confidence (e.g. POSITIVE / NEGATIVE, or a topic class)
  Dataset suggestions (small, public, REAL labels on a held-out split):
    - glue / sst2     : binary sentiment (the C9 dataset; note: test labels hidden, eval on validation)
    - ag_news         : 4-class news topic (World / Sports / Business / Sci-Tech), real test labels
    - emotion         : 6-class tweet emotion, has train/validation/test
  Good when the job is routing / tagging / sentiment / intent.

PATH B - Encoder-decoder Q&A chatbot (C10 style)
  Model:   t5-small
  Output:  a written answer (free text)
  Dataset suggestions (small, public):
    - squad           : reading-comprehension Q&A (the C10 dataset; answer is a span in the context)
  Good when the job must WRITE an answer. Remember: it will hallucinate off-context.

Only run the cells for the path you pick. The two paths never run in the same session.

In [ ]:
# Set this to "A" (encoder-only classifier) or "B" (encoder-decoder Q&A), then run.
# Every later cell checks PATH and runs only if it matches, so the other path's cells are
# no-ops. This lets one notebook serve both shells without variable collisions.
PATH = "A"  # YOUR CHOICE: change to "A" (classifier) or "B" (Q&A) before you start the labs

# Soft gate (PROVIDED - do not edit): a bad value warns instead of crashing, so the notebook
# always reaches the finale. If PATH is not "A" or "B" we fall back to "A" so the run continues.
if PATH not in ("A", "B"):
    print("WARNING: PATH was not 'A' or 'B'; defaulting to 'A' so the notebook still runs.")
    PATH = "A"
print(f"You chose Path {PATH}.")
print("Run only the cells whose header starts with 'Path " + PATH + "'. Skip the other path.")

## Path A: encoder-only classification chatbot

(Skip this whole section if you chose Path B.)

This is the C9 architecture. You will fine-tune distilbert-base-uncased into a text
classifier and wrap it in a chatbot that returns a label + confidence. The pieces are the
same ones you used in C9; here you wire them yourself.

Hyperparameters (sensible defaults from C9 - you may justify changes in your write-up):
- MODEL_NAME = "distilbert-base-uncased"
- MAX_LENGTH = 64        (short support messages; longer wastes compute)
- LR = 2e-5              (standard fine-tune LR for BERT-family classification)
- NUM_EPOCHS = 2
- BATCH_SIZE = 32
- TRAIN_SUBSET = 6000    (subsample so the fine-tune fits a class session on a T4)

Reminder from C9: if your dataset is glue/sst2, the test split labels are hidden (-1), so
you EVALUATE ON THE VALIDATION SPLIT. ag_news and emotion have real test labels.

**Diagram: the Path A fine-tuning loop the Trainer runs for you**

![Path A fine-tuning loop](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/3-Transfer-Chatbot/diagrams/capstone-c/path-a-finetune-loop.png)

### Lab A1 (Path A): load the tokenizer, model, and dataset

Goal: load distilbert-base-uncased as a sequence classifier with the right number of labels,
and load your chosen dataset.

Steps:
1. Load the tokenizer for MODEL_NAME.
2. Decide NUM_LABELS from your dataset (2 for sst2, 4 for ag_news, 6 for emotion) and build
   id2label / label2id dicts so the saved model returns word labels later (as in C9).
3. Load the model for sequence classification with that label count and the label maps.
4. Load the dataset with load_dataset and identify the text field and the label field.

Hints: the tokenizer and the model both come from the same factory classes you imported,
each built from a pretrained name. The text field is "sentence" for sst2 and "text" for
ag_news / emotion; the label field is "label" for all three.

In [ ]:
if PATH == "A":
    MODEL_NAME = "distilbert-base-uncased"
    MAX_LENGTH = 64
    LR = 2e-5
    NUM_EPOCHS = 2
    BATCH_SIZE = 32
    TRAIN_SUBSET = 6000

    # Pick ONE of the suggested sets and describe it to load_dataset. Set the loader args and fields.
    DATASET_ARGS = None   # YOUR CODE: what load_dataset needs to fetch your chosen set
    TEXT_FIELD = None     # YOUR CODE: which column holds the input text
    LABEL_FIELD = None    # YOUR CODE: which column holds the integer label
    NUM_LABELS = None     # YOUR CODE: how many classes your dataset has

    # Build the label maps so reloaded models return word labels (C9 pattern).
    # YOUR CODE: build id2label = {0: "...", 1: "..."} and label2id (the reverse) for your set.
    id2label = None       # YOUR CODE
    label2id = None       # YOUR CODE

    # YOUR CODE: load the tokenizer for MODEL_NAME.
    tokenizer = None
    # YOUR CODE: load the sequence-classification model for MODEL_NAME, passing the label count
    #            and both label maps through so the head has the right width and speaks words.
    model = None

    # Load the dataset once the args are set (skipped while DATASET_ARGS is still None).
    raw = load_dataset(*DATASET_ARGS) if DATASET_ARGS is not None else None

    # Verification (PROVIDED - runs only once you have loaded the model):
    if model is not None:
        assert tokenizer is not None, "Load both tokenizer and model."
        assert model.config.num_labels == NUM_LABELS, "Model label count must match NUM_LABELS."
        assert set(model.config.id2label.keys()) == set(range(NUM_LABELS)), "id2label keys 0..N-1."
        assert TEXT_FIELD in raw["train"].column_names, "TEXT_FIELD not in the dataset."
        print("Path A loaded:", MODEL_NAME, "| labels:", model.config.id2label)

In [ ]:
# ### Lab A2 (Path A, TIER 2): tokenize and assemble the train / eval splits
#
# Turn the raw text into model inputs (the C9 tokenize_fn pattern), then carve out a small
# train subset and a REAL held-out eval split. Two ideas you must combine: the loss reads its
# target from a column named "labels", not "label"; and one of the suggested sets hides its
# test labels, so think about which split is honestly held out for it. The collator is provided.

if PATH == "A" and globals().get("raw") is not None:
    def tokenize_fn(batch):
        return None  # YOUR CODE: turn the text column into model inputs, capped at MAX_LENGTH

    tokenized = None      # YOUR CODE: apply tokenize_fn across the data, then make the target column "labels"

    # YOUR CODE: build train_ds (a reproducibly shuffled TRAIN_SUBSET-sized slice) and a held-out eval_ds.
    train_ds = None
    eval_ds = None

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Verification (PROVIDED - runs only once you have built the splits):
    if train_ds is not None:
        assert "labels" in train_ds.column_names, "Rename the label column to 'labels'."
        assert len(train_ds) == TRAIN_SUBSET, "train_ds must hold exactly TRAIN_SUBSET rows."
        assert len(eval_ds) > 0, "eval_ds must be a non-empty held-out split."
        sample = train_ds[0]
        assert "input_ids" in sample and len(sample["input_ids"]) <= MAX_LENGTH, "Tokenize with max_length."
        print("Path A tokenized. train:", len(train_ds), "eval:", len(eval_ds))

In [ ]:
# ### Lab A3 (Path A): build TrainingArguments and the Trainer
#
# Goal: wire the exact C9 training setup. You provide the argument VALUES; the structure is
# fixed. Use processing_class=tokenizer (the deprecated tokenizer= keyword is gone in 4.57).
#
# Steps:
#   1. Write compute_metrics(eval_pred) that returns accuracy (compare predicted class to gold).
#   2. Fill TrainingArguments: an output dir, the per-device train/eval batch size from BATCH_SIZE,
#      the learning rate from LR, the epoch count from NUM_EPOCHS, and eval_strategy per epoch.
#   3. Construct the Trainer with model, args, train/eval datasets, data_collator,
#      processing_class=tokenizer, and compute_metrics.

if PATH == "A" and globals().get("train_ds") is not None:
    accuracy_metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        # YOUR CODE: turn the per-class scores into a single predicted class, then score it
        #            against the gold labels with accuracy_metric.
        preds = None
        return None  # YOUR CODE: return a dict like {"accuracy": ...}

    training_args = TrainingArguments(
        output_dir="distilbert_capstone",
        per_device_train_batch_size=None,   # YOUR CODE: the training batch size
        per_device_eval_batch_size=None,    # YOUR CODE: the eval batch size
        learning_rate=None,                 # YOUR CODE: the fine-tune learning rate
        num_train_epochs=None,              # YOUR CODE: how many passes over the data
        eval_strategy="epoch",              # provided: per-epoch eval (NOT evaluation_strategy)
        logging_steps=50,
        seed=SEED,
        fp16=(device.type == "cuda"),       # provided: GPU-only mixed precision
        use_cpu=(device.type != "cuda"),    # provided: force CPU off-GPU so Apple-Silicon MPS does not crash
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=data_collator,
        processing_class=tokenizer,   # provided: 4.57 standard; tokenizer= is removed
        compute_metrics=compute_metrics,
    )

    # Verification (PROVIDED - runs only once you have filled the values above):
    if trainer.args.learning_rate is not None:
        assert trainer.args.num_train_epochs == NUM_EPOCHS, "Wire NUM_EPOCHS in."
        assert trainer.args.per_device_train_batch_size == BATCH_SIZE, "Wire BATCH_SIZE in."
        print("Path A Trainer ready.")

In [ ]:
# ### Lab A4 (Path A): fine-tune
#
# One line. Everything was wired in Lab A3. On a Colab T4 this is ~3-6 minutes for
# TRAIN_SUBSET=6000 and 2 epochs.

if PATH == "A" and globals().get("trainer") is not None:
    train_result = None  # YOUR CODE: kick off training and keep what it returns

    # Verification (PROVIDED - runs only once you have launched training):
    if train_result is not None:
        print("Path A training done. Final train loss:", round(train_result.training_loss, 4))

In [ ]:
# ### Lab A5 (Path A): evaluate honestly, then build the chatbot inference function
#
# Goal: get the held-out accuracy and wrap the model as classify_message(text), the exact
# C9 helper the Gradio shell calls.
#
# Steps:
#   1. Score the model on the held-out eval set and read its accuracy.
#   2. Write classify_message(text): turn one string into a single (label, confidence) pair
#      the model is most sure of.

if PATH == "A" and globals().get("trainer") is not None:
    eval_metrics = None  # YOUR CODE: score the model on the held-out split and keep the metrics
    print("Path A held-out metrics:", eval_metrics)

    @torch.no_grad()
    def classify_message(text):
        model.eval()
        # YOUR CODE: turn `text` into model inputs on the right device, run the model, convert the
        #            scores into probabilities, pick the most likely class, and return that class's
        #            word label together with its probability as a float in [0, 1].
        return None  # YOUR CODE: return (label_string, confidence_float)

    # Verification (PROVIDED - runs only once you have evaluated and written classify_message):
    if eval_metrics is not None and classify_message("warmup") is not None:
        assert "eval_accuracy" in eval_metrics, "Evaluation should report eval_accuracy."
        label, conf = classify_message("the support team fixed my issue in minutes, amazing")
        assert isinstance(label, str) and 0.0 <= conf <= 1.0, "Return (label_string, confidence in [0,1])."
        print(f"Sample classification -> {label} ({conf:.2f})")

In [ ]:
# ### Safety-net (Path A) - PROVIDED, do not edit
#
# This capstone is "assemble unaided", so the labs above use hard asserts as gates. If you
# skipped or could not finish a Path A lab, this cell backfills WORKING fallbacks (an untrained
# but runnable classifier and a real classify_message) so the shared save/reload and the Gradio
# ship cells below still run end to end. If you completed the labs, every branch below is a
# no-op (the values already exist), so this never overrides your work.

if PATH == "A":
    if globals().get("tokenizer") is None or globals().get("model") is None:
        print("Safety-net: building a fallback DistilBERT classifier (untrained).")
        _id2label = {0: "NEGATIVE", 1: "POSITIVE"}
        tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
        model = AutoModelForSequenceClassification.from_pretrained(
            "distilbert-base-uncased", num_labels=2,
            id2label=_id2label, label2id={v: k for k, v in _id2label.items()},
        ).to(device)
        if "MAX_LENGTH" not in globals():
            MAX_LENGTH = 64

    if globals().get("classify_message") is None:
        @torch.no_grad()
        def classify_message(text):
            model.eval()
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
            probs = torch.softmax(model(**enc).logits, dim=-1)[0]
            class_id = int(torch.argmax(probs))
            return (model.config.id2label[class_id], float(probs[class_id]))

    # The shared save cell reads trainer.model; provide a minimal stand-in if the Trainer
    # was never built so save_pretrained still has weights to write.
    if globals().get("trainer") is None:
        class _ModelHolder:
            pass
        trainer = _ModelHolder()
        trainer.model = model

    print("Safety-net (Path A) checked: classify_message and trainer.model are ready.")


## Path B: encoder-decoder Q&A chatbot

(Skip this whole section if you chose Path A.)

This is the C10 architecture. You will fine-tune t5-small to ANSWER a question given a
context paragraph, and wrap it in a chatbot with two text inputs (question, context). The
pieces are the same ones you used in C10; here you wire them yourself.

Hyperparameters (sensible defaults from C10 - you may justify changes in your write-up):
- MODEL_NAME = "t5-small"
- max_input = 256        (question + context fit comfortably)
- max_target = 32        (SQuAD answers are short spans)
- LR = 3e-4              (T5 fine-tunes at a HIGHER LR than BERT classification, ~3e-4)
- NUM_EPOCHS = 2
- BATCH_SIZE = 16
- small_train = 2000     (subsample so the fine-tune fits a class session on a T4)
- small_val = 200

Honesty note carried from C10: a SQuAD-trained T5 mostly COPIES a span from the context.
Ask it something the context does not answer and it will confidently invent an answer.
SQuAD v1 never teaches "no answer", so off-context hallucination is EXPECTED, not a bug.
The fix is grounding (RAG); you will probe this in the evaluation cell.

### Lab B1 (Path B): load t5-small, load SQuAD, and build the T5 input string

Goal: load the seq2seq model + tokenizer, load SQuAD, and write build_input(question,
context) exactly as in C10 so T5 sees the task it was trained on.

Steps:
1. Load the tokenizer and the seq2seq model from MODEL_NAME (use the seq2seq Auto* factory,
   not the classification one).
2. Load squad with load_dataset; note each example has question, context, and answers
   (a dict with a "text" LIST and an "answer_start" LIST - take the FIRST gold answer).
3. Write build_input(question, context) returning the C10 text-to-text prompt that frames the
   question alongside the context (T5 needs the task framed as text-to-text).

Hint: the gold answer for an example is the first element of its answers["text"] list.

In [ ]:
# ### Lab B1 / B-preprocess (Path B)
if PATH == "B":
    MODEL_NAME = "t5-small"
    max_input = 256    # caps question + context. CAVEAT: a long SQuAD context can be truncated
                       # here, which may cut off the gold answer span; raise this if you see
                       # answers going missing on long passages (at the cost of more compute).
    max_target = 32
    LR = 3e-4
    NUM_EPOCHS = 2
    BATCH_SIZE = 16
    small_train = 2000
    small_val = 200

    # YOUR CODE: load the tokenizer for MODEL_NAME.
    tokenizer = None
    # YOUR CODE: load the seq2seq model for MODEL_NAME.
    model = None

    raw = load_dataset("squad")

    def build_input(question, context):
        # YOUR CODE: frame the task as the C10 text-to-text prompt that pairs the question
        #            with the context so T5 sees the format it was trained on.
        return None

    def preprocess(examples):
        # PROVIDED scaffold; you fill the two tokenizer calls.
        inputs = [build_input(q, c) for q, c in zip(examples["question"], examples["context"])]
        targets = [a["text"][0] for a in examples["answers"]]   # first gold answer (C10 gotcha)
        # YOUR CODE: tokenize the source `inputs`, truncating to max_input -> model_inputs
        model_inputs = None
        # YOUR CODE: tokenize the gold `targets` as the decoder side and attach them as the
        #            "labels" the loss reads.
        return model_inputs

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)  # pads labels with -100

    # Run the preprocessing once the tokenizer is loaded (skipped while it is still None):
    if tokenizer is not None and build_input("Q?", "C.") is not None:
        small = raw["train"].shuffle(seed=SEED)
        tokenized_train = small.select(range(small_train)).map(
            preprocess, batched=True, remove_columns=small.column_names)
        tokenized_val = raw["validation"].shuffle(seed=SEED).select(range(small_val)).map(
            preprocess, batched=True, remove_columns=raw["validation"].column_names)

        # Verification (PROVIDED - do not edit):
        assert build_input("Q?", "C.") == "question: Q?  context: C.", "build_input format must match C10."
        assert "labels" in tokenized_train.column_names, "preprocess must attach 'labels' via text_target."
        assert len(tokenized_train) == small_train and len(tokenized_val) == small_val, "Subset sizes."
        print("Path B preprocessed. train:", len(tokenized_train), "val:", len(tokenized_val))

### Lab B2 (Path B): configure Seq2SeqTrainingArguments and Seq2SeqTrainer

Goal: wire the C10 seq2seq training setup. T5 specifics you must respect:
- predict_with_generate must be on so eval uses model.generate() (real decoding, not teacher forcing).
- fp16 only when running on a GPU (device.type == "cuda").
- Use the HIGHER T5 learning rate, not the BERT classification LR (recall what C10 used for T5).
- Standardize on processing_class=tokenizer for the Seq2SeqTrainer too (C10 used the
  deprecated tokenizer= keyword; in 4.57 prefer processing_class= for both trainers).
- Never build decoder_input_ids yourself; teacher forcing is automatic.

You provide the argument VALUES; the structure is fixed.

**Diagram: the Path B fine-tuning loop the Seq2SeqTrainer runs**

![Path B fine-tuning loop](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/3-Transfer-Chatbot/diagrams/capstone-c/path-b-finetune-loop.png)

In [ ]:
if PATH == "B" and globals().get("tokenized_train") is not None:
    training_args = Seq2SeqTrainingArguments(
        output_dir="t5_capstone_qa",
        per_device_train_batch_size=None,   # YOUR CODE: the training batch size
        per_device_eval_batch_size=None,    # YOUR CODE: the eval batch size
        learning_rate=None,                 # YOUR CODE: the T5 fine-tune LR (higher than BERT's)
        num_train_epochs=None,              # YOUR CODE: how many passes over the data
        eval_strategy="epoch",              # provided (NOT evaluation_strategy)
        predict_with_generate=None,         # YOUR CODE: turn this on so eval truly decodes
        fp16=(device.type == "cuda"),       # provided: GPU-only mixed precision
        use_cpu=(device.type != "cuda"),    # provided: force CPU off-GPU so Apple-Silicon MPS does not crash
        logging_steps=50,
        seed=SEED,
        report_to="none",
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        processing_class=tokenizer,   # provided: standardized across both trainers in 4.57
    )

    # Verification + training (PROVIDED - runs only once you have filled the values above):
    if training_args.learning_rate is not None and training_args.predict_with_generate:
        assert training_args.predict_with_generate is True, "Turn predict_with_generate on for seq2seq eval."
        assert training_args.learning_rate > 1e-4, "T5 wants a higher LR than BERT classification."
        train_result = trainer.train()
        print("Path B training done. Final train loss:", round(train_result.training_loss, 4))

In [ ]:
# ### Lab B3 (Path B, TIER 3 - no scaffolding): write answer_question(question, context)
#
# This is the finale lab. There are no steps, no inline hints, and no verification block.
# Implement the C10 inference helper the Gradio shell calls, from a blank function body.
# Given a question and a context paragraph, it must return the model's written answer as a
# single string. Everything you need you have already used in C10 and earlier in this path.

if PATH == "B":
    # YOUR CODE: define answer_question(question, context) that returns the model's written
    # answer as a single string. Leave it as None below until you implement it; the safety-net
    # cell that follows will backfill a working version so the finale still ships.
    answer_question = None

<details>
<summary>Stuck on Lab B3? Reveal the safety-net</summary>

```python
@torch.no_grad()
def answer_question(question, context):
    model.eval()
    text = build_input(question, context)
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input).to(device)
    out = model.generate(**enc, max_new_tokens=max_target)
    return tokenizer.decode(out[0], skip_special_tokens=True)
```
</details>

In [ ]:
# ### Safety-net (Path B) - PROVIDED, do not edit
#
# Lab B3 above is a bare TIER-3 lab with no verification, and the earlier Path B labs gate on
# hard asserts. If you skipped or could not finish a Path B lab, this cell backfills WORKING
# fallbacks (an untrained but runnable t5-small, build_input, and a real answer_question) so the
# Lab B4 probe, the shared save/reload, and the Gradio ship below all still run. If you finished
# the labs, every branch is a no-op and your work is left untouched.

if PATH == "B":
    if "max_input" not in globals():
        max_input = 256
    if "max_target" not in globals():
        max_target = 32

    if globals().get("tokenizer") is None or globals().get("model") is None:
        print("Safety-net: building a fallback t5-small (untrained).")
        tokenizer = AutoTokenizer.from_pretrained("t5-small")
        model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(device)

    # Rebuild build_input if it is missing OR if the unfinished lab left it returning None.
    _bi = globals().get("build_input")
    if _bi is None or _bi("Q?", "C.") is None:
        def build_input(question, context):
            return f"question: {question}  context: {context}"

    # Rebuild answer_question if it is missing OR if the unfinished lab left it returning None.
    _aq = globals().get("answer_question")
    if _aq is None or _aq("Who?", "A founded B.") is None:
        @torch.no_grad()
        def answer_question(question, context):
            model.eval()
            enc = tokenizer(build_input(question, context), return_tensors="pt",
                            truncation=True, max_length=max_input).to(device)
            out = model.generate(**enc, max_new_tokens=max_target)
            return tokenizer.decode(out[0], skip_special_tokens=True)

    # The shared save cell reads trainer.model; provide a minimal stand-in if the Seq2SeqTrainer
    # was never built so save_pretrained still has weights to write.
    if globals().get("trainer") is None:
        class _ModelHolder:
            pass
        trainer = _ModelHolder()
        trainer.model = model

    print("Safety-net (Path B) checked: build_input, answer_question and trainer.model are ready.")

In [ ]:
# ### Lab B4 (Path B): qualitative spot-check + the hallucination probe
#
# Goal: read a few fine-tuned answers next to the gold spans (does the model land the right
# answer when the context contains it?), then deliberately feed an OFF-CONTEXT question and
# watch T5 confidently invent an answer. This is the honest evaluation leadership asked for.
# (We compare to the GOLD answers, not to an untrained baseline - the point is to judge the
# fine-tuned model's answers against ground truth and expose its failure mode.)
#
# Steps:
#   1. Pick 3 validation examples; for each, print question, gold answer, and answer_question(...).
#   2. Run the provided hallucination probe: a real question whose context does NOT contain
#      the answer. Read the output critically - it will likely fabricate.

if PATH == "B":
    val = raw["validation"]
    for i in range(3):
        ex = val[i]
        gold = ex["answers"]["text"][0]
        # YOUR CODE: ask the model this example's own question over its own context -> pred
        pred = None
        print(f"Q: {ex['question']}\n  gold: {gold}\n  model: {pred}\n")

    # Hallucination probe (PROVIDED - do not edit). The context is about cats; the question
    # is about a CEO, which the context cannot answer. Watch it invent something anyway.
    probe_ctx = "Cats are small carnivorous mammals often kept as pets. They sleep most of the day."
    probe_q = "Who is the CEO of the company mentioned in the text?"
    print("OFF-CONTEXT PROBE")
    print("  context:", probe_ctx)
    print("  question:", probe_q)
    print("  model:", answer_question(probe_q, probe_ctx))
    print("\nNote for your write-up: SQuAD v1 never teaches 'no answer', so the model copies or")
    print("invents. The production fix is grounding (RAG) + an abstention threshold.")

**Diagram: save, reload, and ship the guarded Gradio chatbot**

![Save, reload, and ship the Gradio chatbot](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/3-Transfer-Chatbot/diagrams/capstone-c/save-reload-gradio-chatbot.png)

In [ ]:
# ### Save and reload (runs for whichever path you chose)
#
# save_pretrained writes BOTH the model and the tokenizer; reload BOTH. id2label travels in
# config.json, so a reloaded Path A model returns word labels. After reload we REBIND the
# global `model` and `tokenizer` to the on-disk objects so the inference helpers use them
# (C10 continuity: helpers close over these globals).

SAVE_DIR = "capstone_model_A" if PATH == "A" else "capstone_model_B"

# YOUR CODE: write the trained weights to SAVE_DIR (persist the trainer's model, not just the
#            original handle), and write the tokenizer to the same folder so reload finds both.
# (Until you fill these, the safety-net below saves a runnable fallback so the finale still ships.)

# Safety-net (PROVIDED - do not edit): if nothing was saved above, persist the current
# trainer.model + tokenizer so the reload and the Gradio ship below still run end to end.
import os
if not os.path.isdir(SAVE_DIR):
    trainer.model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)

# Reload BOTH from disk (PROVIDED - do not edit):
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
if PATH == "A":
    model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR).to(device)

# Verification (PROVIDED - do not edit):
assert os.path.isdir(SAVE_DIR), "Save the model + tokenizer to SAVE_DIR first."
if PATH == "A":
    label, conf = classify_message("the agent was rude and slow")
    print("Reloaded Path A still works ->", label, round(conf, 2))
else:
    print("Reloaded Path B still works ->",
          answer_question("Who founded Data Trainers?",
                          "Data Trainers LLC was founded by Axel Sirota."))
print("Saved + reloaded from", SAVE_DIR)

In [ ]:
# ### Ship it: the guarded Gradio chatbot (runs for whichever path you chose)
#
# This is the C9/C10 Gradio guard, verbatim: try/except ImportError so a no-Gradio env still
# works, share=True for the Colab public link, and a non-Gradio fallback that calls inference
# directly. Path A is a single text box -> label + confidence; Path B is two boxes
# (question, context) -> written answer.

try:
    import gradio as gr

    if PATH == "A":
        def gradio_fn(message):
            label, conf = classify_message(message)
            return f"{label} (confidence {conf:.2f})"

        demo = gr.Interface(
            fn=gradio_fn,
            inputs=gr.Textbox(label="Customer message"),
            outputs=gr.Textbox(label="Prediction"),
            title="Capstone C - DistilBERT ticket classifier",
        )
    else:
        def gradio_fn(question, context):
            return answer_question(question, context)

        demo = gr.Interface(
            fn=gradio_fn,
            inputs=[gr.Textbox(label="Question"), gr.Textbox(label="Context / help-doc snippet")],
            outputs=gr.Textbox(label="Answer"),
            title="Capstone C - T5 Q&A assistant",
        )

    # share=True publishes a temporary public Colab link (it is the default in Colab anyway).
    demo.launch(share=True)

except ImportError:
    # Non-Gradio fallback: call inference directly so the notebook still demonstrates the model.
    print("Gradio not installed - direct inference fallback:")
    if PATH == "A":
        print(classify_message("this product is fantastic, thank you"))
    else:
        print(answer_question("Who founded Data Trainers?",
                              "Data Trainers LLC was founded by Axel Sirota."))

In [ ]:
# ### Your write-up (fill the strings, then run)
#
# This is the deliverable leadership asked for. Be honest about failure modes.
#   - Technical memo: dataset + why, metric + number, the single biggest failure mode,
#     and one concrete production next step.
#   - Non-technical pitch: 3 sentences a product manager would understand.
#
# Path A failure modes to consider: confident-but-wrong on sarcasm / negation, class
# imbalance, domain drift. Path B failure modes: span-copying, off-context hallucination,
# no "I don't know". Production levers: confidence-threshold abstention, drift monitoring,
# RAG grounding (Path B), periodic re-evaluation.

DATASET_CHOICE = None       # YOUR CODE: which dataset you used and one line on WHY it fit
HEADLINE_METRIC = None      # YOUR CODE: your held-out number (e.g. "accuracy 0.91" or "qualitative")
BIGGEST_FAILURE = None      # YOUR CODE: the single worst failure mode you observed
PRODUCTION_NEXT_STEP = None # YOUR CODE: one concrete thing you would add before shipping

PITCH_1 = None  # YOUR CODE: what the bot does, in plain language
PITCH_2 = None  # YOUR CODE: one honest limitation a customer might hit
PITCH_3 = None  # YOUR CODE: the business value if the limitation is managed

# Verification (PROVIDED - runs once every field is filled with a real sentence):
fields = [("DATASET_CHOICE", DATASET_CHOICE), ("HEADLINE_METRIC", HEADLINE_METRIC),
          ("BIGGEST_FAILURE", BIGGEST_FAILURE), ("PRODUCTION_NEXT_STEP", PRODUCTION_NEXT_STEP),
          ("PITCH_1", PITCH_1), ("PITCH_2", PITCH_2), ("PITCH_3", PITCH_3)]
if all(isinstance(v, str) and len(v) > 0 for _, v in fields):
    print("TECHNICAL MEMO")
    print(" dataset:", DATASET_CHOICE)
    print(" metric :", HEADLINE_METRIC)
    print(" failure:", BIGGEST_FAILURE)
    print(" next   :", PRODUCTION_NEXT_STEP)
    print("\nPITCH:", PITCH_1, PITCH_2, PITCH_3)
else:
    print("Write-up not submitted yet: fill all seven strings above, then re-run this cell.")

## Self-check (answer before you submit - no code needed)

1. You chose Path ___. Why was an encoder-only classifier (or encoder-decoder Q&A) the right
   architecture for your task? Name one task where the OTHER path would have been correct.
2. Why do we evaluate on a held-out split, and why is glue/sst2's test split unusable for
   reporting accuracy?
3. (Path B) Explain in one sentence why a SQuAD-trained T5 hallucinates on an off-context
   question, and name the production fix.
4. Why does id2label need to live in config.json for the reloaded chatbot to return word
   labels?
5. Name one production concern (monitoring, drift, cost, or abstention) and how you would
   address it for YOUR bot.

## Homework extensions (async, deeper)

- STRETCH (in-class, fast finishers): Path A - freeze the DistilBERT backbone and train only
  the classification head; report how much accuracy you traded for the speed-up. Path B -
  sweep two decoding settings (greedy vs num_beams=4) on five questions and describe the
  difference.
- HOMEWORK 1 (both): add a confidence-threshold abstention layer - if Path A's confidence is
  below a threshold (or Path B's answer is empty / low-probability), return "I am not sure,
  routing to a human." Measure coverage vs error on the held-out set.
- HOMEWORK 2 (Path B): ground the model with RAG - reuse an embedder to retrieve the most
  relevant help-doc snippet as the context, then answer over it. Re-run the off-context probe
  and show grounding removes the hallucination.
- HOMEWORK 3 (both): compute a proper metric (accuracy + macro-F1 for A via evaluate; EM/F1
  for B via the squad metric) and write a one-paragraph drift-monitoring plan.

## That is the course - what you learned

You went from tools-first NLP, to embeddings as geometry, to an MLP on word2vec, to
fine-tuning real transformers, and now to a shipped, evaluated, honestly-documented chatbot
that YOU built end to end. That is the whole job.

**Diagram: RAG grounding - the production fix for hallucination**

![RAG grounding pipeline](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/3-Transfer-Chatbot/diagrams/capstone-c/rag-grounding.png)